In [ ]:
# Cell 1 — clone/pull + install
!git clone https://github.com/Jaswanth-K1210/SDAM.git 2>/dev/null || (cd SDAM && git pull --no-edit origin main)
%cd SDAM
!pip install -q -r probe/requirements.txt
print("Setup complete")

In [ ]:
# Cell 2 — GPU check
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — CLEVR v1.0 (only if not present; reuses cached features if the probe/objectness run left them)
import os
if not os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"):
    os.makedirs("data", exist_ok=True)
    !wget -q --show-progress https://dl.fbaipublicfiles.com/clevr/CLEVR_v1.0.zip -O data/CLEVR_v1.0.zip
    !cd data && unzip -q CLEVR_v1.0.zip && rm -f CLEVR_v1.0.zip
print("scenes:", os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"))

In [ ]:
# Cell 4 — pure-unit test suites must be GREEN first (factoring math + verdict logic)
!python -m pytest tests/test_factored.py tests/test_factored_verdict.py tests/test_variance.py tests/test_objectness_pipeline.py -q

In [ ]:
# Cell 5 — run FactoredSDAM pipeline (verdict at variance-matched w=1.0x; 0.5x/2x robustness)
!python -m experiments.factored_pipeline

In [ ]:
# Cell 6 — Phase 1 plot (factored vs objectness/random/zeroed across w)
from IPython.display import Image, display
display(Image("results/factored_phase1.png"))

In [ ]:
# Cell 7 — full results JSON
print(open("results/factored_pipeline.json").read())

In [ ]:
# Cell 8 — commit results to GitHub
from google.colab import userdata
import os
try: token = userdata.get("GITHUB_TOKEN")
except Exception: token = ""
!git config --global user.email "jaswanthkoppisetty@gmail.com"
!git config --global user.name "Jaswanth-K1210"
!mkdir -p results_archive/factored
!cp -f results/factored_pipeline.json results/factored_phase1.png results_archive/factored/ 2>/dev/null || true
!git add results_archive/factored/
!git commit -m "results(factored): FactoredSDAM Phase 1 + w-robustness run" || echo "nothing to commit"
if token:
    os.system(f"git remote set-url origin https://{token}@github.com/Jaswanth-K1210/SDAM.git")
    os.system("git push origin main"); print("Pushed.")
else:
    print("Add GITHUB_TOKEN to Colab Secrets then re-run.")